# Data Preparation

## What this notebook does
1. Loads the VAE latent vectors (β = 3×10⁻⁵) produced by the training pipeline  
2. Loads and preprocesses PPMI clinical assessment tables  
3. Computes UPDRS subscores and total scores  
4. Merges clinical data with latent vectors on `PATNO + EVENT_ID`  
5. Applies a **patient-stratified** train/val split (fixes data leakage in baseline)  
6. Fits the SBR PCA **once** on the training set and saves it for reuse  
7. Saves the final train and validation files  

## Improvements over baseline (Mahmoud's 5.0)
- Patient-stratified split: all visits of one patient stay in one split  
- Clinical scores added: UPDRS I–IV, MoCA, disease duration, REM sleep  
- SBR PCA fitted once and saved — not recomputed in every notebook  
- Explicit coverage report: how many rows have each clinical variable  


## 1. Imports and Configuration

In [2]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
import glob
warnings.filterwarnings('ignore')

# Paths
# Latent vectors (VAE output — beta = 3e-5)
LATENT_FILE = '../../data/baseline/final_train_combined_vae_data.csv'

# Raw PPMI clinical tables
CLINICAL_DIR = '../../data/raw/ppmi_clinical'

UPDRS_I_FILE   = os.path.join(CLINICAL_DIR, 'MDS_UPDRS_Part_I__Patient_Questionnaire-Archived_25Jun2026.csv')
UPDRS_II_FILE  = os.path.join(CLINICAL_DIR, 'MDS_UPDRS_Part_II__Patient_Questionnaire-Archived_25Jun2026.csv')
UPDRS_III_FILE = os.path.join(CLINICAL_DIR, 'MDS_UPDRS_Part_III-Archived_25Jun2026.csv')
UPDRS_IV_FILE  = os.path.join(CLINICAL_DIR, 'MDS_UPDRS_Part_IV-Archived_25Jun2026.csv')
MOCA_FILE      = os.path.join(CLINICAL_DIR, 'Montreal_Cognitive_Assessment__MoCA_-Archived_25Jun2026.csv')
PD_FEAT_FILE   = os.path.join(CLINICAL_DIR, 'PD_Features-Archived_25Jun2026.csv')
REM_FILE       = os.path.join(CLINICAL_DIR, 'REM_Sleep_Disorder_Questionnaire-Archived_25Jun2026.csv')

# Output paths
TRAIN_OUT  = '../../data/processed/clinical_merged/train.csv'
VAL_OUT    = '../../data/processed/clinical_merged/val.csv'
SCALER_OUT = '../../results/models/scaler_sbr.pkl'
PCA_OUT    = '../../results/models/pca_sbr.pkl'

# Parameters
TRAIN_RATIO        = 0.8
RANDOM_STATE       = 42
VARIANCE_THRESHOLD = 0.1   # latent dims with std below this treated as collapsed
SBR_COLS = [
    'DATSCAN_CAUDATE_R', 'DATSCAN_CAUDATE_L',
    'DATSCAN_PUTAMEN_R', 'DATSCAN_PUTAMEN_L',
    'DATSCAN_PUTAMEN_R_ANT', 'DATSCAN_PUTAMEN_L_ANT'
]
N_SBR_PCS   = 3
PATIENT_COL = 'PATNO'
LABEL_COL   = 'label'
JOIN_KEY    = ['PATNO', 'EVENT_ID']

# Create output directories
os.makedirs('../../data/processed/clinical_merged', exist_ok=True)
os.makedirs('../../results/models', exist_ok=True)

print("Configuration loaded.")
print(f"  Train ratio:        {TRAIN_RATIO}")
print(f"  Random state:       {RANDOM_STATE}")
print(f"  SBR columns:        {len(SBR_COLS)}")
print(f"  SBR PCs to extract: {N_SBR_PCS}")


# Check if the clinical data files exist

tables = {
    "MDS_UPDRS_Part_I":              UPDRS_I_FILE,
    "MDS_UPDRS_Part_II":             UPDRS_II_FILE,
    "MDS_UPDRS_Part_III":            UPDRS_III_FILE,
    "MDS_UPDRS_Part_IV":             UPDRS_IV_FILE,
    "Montreal_Cognitive_Assessment": MOCA_FILE,
    "PD_Features":                   PD_FEAT_FILE,
    "REM_Sleep":                     REM_FILE,
}

print("-" * 65)     
print("Checking for clinical data files...")
for name, pattern in tables.items():
    matches = glob.glob(pattern)
    if matches:
        df = pd.read_csv(matches[0], nrows=2)
        print(f" File:    {matches[0]}")
    else:
        print(f"✗ {name} — not found")
        print()

Configuration loaded.
  Train ratio:        0.8
  Random state:       42
  SBR columns:        6
  SBR PCs to extract: 3
-----------------------------------------------------------------
Checking for clinical data files...
 File:    ../../data/raw/ppmi_clinical/MDS_UPDRS_Part_I__Patient_Questionnaire-Archived_25Jun2026.csv
 File:    ../../data/raw/ppmi_clinical/MDS_UPDRS_Part_II__Patient_Questionnaire-Archived_25Jun2026.csv
 File:    ../../data/raw/ppmi_clinical/MDS_UPDRS_Part_III-Archived_25Jun2026.csv
 File:    ../../data/raw/ppmi_clinical/MDS_UPDRS_Part_IV-Archived_25Jun2026.csv
 File:    ../../data/raw/ppmi_clinical/Montreal_Cognitive_Assessment__MoCA_-Archived_25Jun2026.csv
 File:    ../../data/raw/ppmi_clinical/PD_Features-Archived_25Jun2026.csv
 File:    ../../data/raw/ppmi_clinical/REM_Sleep_Disorder_Questionnaire-Archived_25Jun2026.csv


## 2. Load Latent Vector

In [3]:
# Load the latent vectors from the baseline(Mahmoud's) VAE model
df_latents = pd.read_csv(LATENT_FILE)

print(f"Shape:           {df_latents.shape}")
print(f"Unique patients: {df_latents[PATIENT_COL].nunique()}")
print(f"Total rows:      {len(df_latents)}")
print(f"\nLabel counts:")
print(df_latents[LABEL_COL].value_counts().to_string())
print(f"\nEVENT_ID distribution:")
print(df_latents['EVENT_ID'].value_counts().to_string())
print(f"\nRows per patient (mean): {len(df_latents) / df_latents[PATIENT_COL].nunique():.2f}")

Shape:           (2373, 303)
Unique patients: 1437
Total rows:      2373

Label counts:
label
PD         2030
Control     233
SWEDD       110

EVENT_ID distribution:
EVENT_ID
SC     1228
V06     392
V04     385
V10     256
U01      41
ST       32
V02      23
V05      10
U02       6

Rows per patient (mean): 1.65


## 3. Load and Preprocess Clinical Tables

### 3.1 MDS-UPDRS Part I — Non-motor symptoms

In [4]:
df_updrs1 = pd.read_csv(UPDRS_I_FILE)

# UPDRS Part I items
UPDRS1_ITEMS = ['NP1SLPN', 'NP1SLPD', 'NP1PAIN', 'NP1URIN',
                'NP1CNST', 'NP1LTHD', 'NP1FATG']

# Compute total score (sum of items, require at least 5 of 7 non-null)
df_updrs1['UPDRS1_TOTAL'] = df_updrs1[UPDRS1_ITEMS].sum(axis=1, min_count=5)
# print(df_updrs1['UPDRS1_TOTAL'].head(2))

df_updrs1 = df_updrs1[JOIN_KEY + ['UPDRS1_TOTAL']].dropna(subset=['UPDRS1_TOTAL'])

print(f"UPDRS-I loaded:  {df_updrs1.shape[0]} rows, {df_updrs1[PATIENT_COL].nunique()} patients")
print(f"Score range:     {df_updrs1['UPDRS1_TOTAL'].min():.0f} – {df_updrs1['UPDRS1_TOTAL'].max():.0f}")
print(f"Mean ± std:      {df_updrs1['UPDRS1_TOTAL'].mean():.1f} ± {df_updrs1['UPDRS1_TOTAL'].std():.1f}")

UPDRS-I loaded:  13905 rows, 2136 patients
Score range:     0 – 25
Mean ± std:      5.2 ± 4.0


### 3.2 MDS-UPDRS Part II - Motor daily living


In [16]:
df_updrs2 = pd.read_csv(UPDRS_II_FILE)

UPDRS2_ITEMS = ['NP2SPCH', 'NP2SALV', 'NP2SWAL', 'NP2EAT', 'NP2DRES',
                'NP2HYGN', 'NP2HWRT', 'NP2HOBB', 'NP2TURN', 'NP2TRMR',
                'NP2RISE', 'NP2WALK', 'NP2FREZ']

df_updrs2['UPDRS2_TOTAL'] = df_updrs2[UPDRS2_ITEMS].sum(axis=1, min_count=10)

df_updrs2 = df_updrs2[JOIN_KEY + ['UPDRS2_TOTAL']].dropna(subset=['UPDRS2_TOTAL'])


print(f"UPDRS-II loaded: {df_updrs2.shape[0]} rows, {df_updrs2[PATIENT_COL].nunique()} patients")
print(f"Score range:     {df_updrs2['UPDRS2_TOTAL'].min():.0f} – {df_updrs2['UPDRS2_TOTAL'].max():.0f}")
print(f"Mean ± std:      {df_updrs2['UPDRS2_TOTAL'].mean():.1f} ± {df_updrs2['UPDRS2_TOTAL'].std():.1f}")

UPDRS-II loaded: 13904 rows, 2136 patients
Score range:     0 – 48
Mean ± std:      5.9 ± 6.4


### 3.3 MDS-UPDRS Part III - Motor examination (most important)

In [35]:
df_updrs3 = pd.read_csv(UPDRS_III_FILE)

UPDRS3_ITEMS = [
    'NP3SPCH', 'NP3FACXP', 'NP3RIGN', 'NP3RIGRU', 'NP3RIGLU',
    'PN3RIGRL', 'NP3RIGLL', 'NP3FTAPR', 'NP3FTAPL', 'NP3HMOVR',
    'NP3HMOVL', 'NP3PRSPR', 'NP3PRSPL', 'NP3TTAPR', 'NP3TTAPL',
    'NP3LGAGR', 'NP3LGAGL', 'NP3RISNG', 'NP3GAIT', 'NP3FRZGT',
    'NP3PSTBL', 'NP3POSTR', 'NP3BRADY', 'NP3PTRMR', 'NP3PTRML',
    'NP3KTRMR', 'NP3KTRML', 'NP3RTARU', 'NP3RTALU', 'NP3RTARL',
    'NP3RTALL', 'NP3RTALJ', 'NP3RTCON'
]

df_updrs3['UPDRS3_TOTAL'] = df_updrs3[UPDRS3_ITEMS].sum(axis=1, min_count=30)

df_updrs3 = df_updrs3[JOIN_KEY + ['UPDRS3_TOTAL', 'NHY']].dropna(subset=['UPDRS3_TOTAL'])
df_updrs3 = df_updrs3.rename(columns={'NHY': 'HOEHN_YAHR'})

print(f"UPDRS-III loaded: {df_updrs3.shape[0]} rows, {df_updrs3[PATIENT_COL].nunique()} patients")
print(f"Score range:      {df_updrs3['UPDRS3_TOTAL'].min():.0f} – {df_updrs3['UPDRS3_TOTAL'].max():.0f}")
print(f"Mean ± std:       {df_updrs3['UPDRS3_TOTAL'].mean():.1f} ± {df_updrs3['UPDRS3_TOTAL'].std():.1f}")
print(f"\nHoehn & Yahr distribution:")
print(df_updrs3['HOEHN_YAHR'].value_counts().sort_index().to_string())

UPDRS-III loaded: 15817 rows, 2134 patients
Score range:      0 – 100
Mean ± std:       17.5 ± 14.6

Hoehn & Yahr distribution:
HOEHN_YAHR
0.0    4802
1.0    2691
2.0    7456
3.0     732
4.0      99
5.0      31


### 3.4 MDS-UPDRS Part IV - Motor complications

In [36]:
df_updrs4 = pd.read_csv(UPDRS_IV_FILE)

UPDRS4_ITEMS = ['NP4WDYSK', 'NP4DYSKI', 'NP4OFF', 'NP4FLCTI', 'NP4FLCTX', 'NP4DYSTN']

df_updrs4['UPDRS4_TOTAL'] = df_updrs4[UPDRS4_ITEMS].sum(axis=1, min_count=5)
df_updrs4 = df_updrs4[JOIN_KEY + ['UPDRS4_TOTAL']].dropna(subset=['UPDRS4_TOTAL'])

print(f"UPDRS-IV loaded: {df_updrs4.shape[0]} rows, {df_updrs4[PATIENT_COL].nunique()} patients")
print(f"Score range:     {df_updrs4['UPDRS4_TOTAL'].min():.0f} – {df_updrs4['UPDRS4_TOTAL'].max():.0f}")
print(f"Mean ± std:      {df_updrs4['UPDRS4_TOTAL'].mean():.1f} ± {df_updrs4['UPDRS4_TOTAL'].std():.1f}")

UPDRS-IV loaded: 5696 rows, 916 patients
Score range:     0 – 17
Mean ± std:      1.8 ± 2.8


### 3.5 Montreal Cognitive Assessment (MoCA)

In [38]:
df_moca = pd.read_csv(MOCA_FILE)

# MCATOT is the total score (0-30, higher = better cognition)
df_moca = df_moca[JOIN_KEY + ['MCATOT']].dropna(subset=['MCATOT'])
df_moca = df_moca.rename(columns={'MCATOT': 'MOCA_TOTAL'})

print(f"MoCA loaded:  {df_moca.shape[0]} rows, {df_moca[PATIENT_COL].nunique()} patients")
print(f"Score range:  {df_moca['MOCA_TOTAL'].min():.0f} – {df_moca['MOCA_TOTAL'].max():.0f}")
print(f"Mean ± std:   {df_moca['MOCA_TOTAL'].mean():.1f} ± {df_moca['MOCA_TOTAL'].std():.1f}")

MoCA loaded:  7854 rows, 2177 patients
Score range:  0 – 30
Mean ± std:   26.6 ± 3.1


### 3.6 PD Features - Disease duration

In [40]:
df_pdfeat = pd.read_csv(PD_FEAT_FILE)

# PDDXDT = diagnosis date (MM/YYYY), INFODT = visit date
# Compute disease duration in years at time of visit
df_pdfeat['PDDXDT']  = pd.to_datetime(df_pdfeat['PDDXDT'],  format='%m/%Y', errors='coerce')
df_pdfeat['INFODT']  = pd.to_datetime(df_pdfeat['INFODT'],  format='%m/%Y', errors='coerce')
df_pdfeat['DISEASE_DURATION_YRS'] = (
    (df_pdfeat['INFODT'] - df_pdfeat['PDDXDT']).dt.days / 365.25
)

# Keep dominant side of symptoms
df_pdfeat = df_pdfeat[JOIN_KEY + ['DISEASE_DURATION_YRS', 'DOMSIDE']].dropna(subset=['DISEASE_DURATION_YRS'])
df_pdfeat = df_pdfeat[df_pdfeat['DISEASE_DURATION_YRS'] >= 0]  # remove negative durations

print(f"PD Features loaded:   {df_pdfeat.shape[0]} rows, {df_pdfeat[PATIENT_COL].nunique()} patients")
print(f"Duration range:       {df_pdfeat['DISEASE_DURATION_YRS'].min():.1f} – {df_pdfeat['DISEASE_DURATION_YRS'].max():.1f} years")
print(f"Mean ± std:           {df_pdfeat['DISEASE_DURATION_YRS'].mean():.1f} ± {df_pdfeat['DISEASE_DURATION_YRS'].std():.1f} years")

PD Features loaded:   1063 rows, 1063 patients
Duration range:       0.0 – 29.5 years
Mean ± std:           2.9 ± 4.6 years


### 3.7 REM Sleep Disorder


In [41]:
df_rem = pd.read_csv(REM_FILE)

# REM sleep behaviour disorder items (yes/no questions, scored 0/1)
REM_ITEMS = ['DRMVIVID', 'DRMAGRAC', 'DRMNOCTB', 'SLPLMBMV', 'SLPINJUR',
             'DRMVERBL', 'DRMFIGHT', 'DRMUMV', 'DRMOBJFL', 'MVAWAKEN']

df_rem['RBD_SCORE'] = df_rem[REM_ITEMS].sum(axis=1, min_count=8)
df_rem = df_rem[JOIN_KEY + ['RBD_SCORE']].dropna(subset=['RBD_SCORE'])

print(f"REM sleep loaded: {df_rem.shape[0]} rows, {df_rem[PATIENT_COL].nunique()} patients")
print(f"Score range:      {df_rem['RBD_SCORE'].min():.0f} – {df_rem['RBD_SCORE'].max():.0f}")
print(f"Mean ± std:       {df_rem['RBD_SCORE'].mean():.1f} ± {df_rem['RBD_SCORE'].std():.1f}")

REM sleep loaded: 8833 rows, 1929 patients
Score range:      0 – 10
Mean ± std:       2.6 ± 2.5


### 4. Merge Clinical Tables with Latent Vectors

In [51]:
df = df_latents.copy()
n_start = len(df)

clinical_tables = {
    'UPDRS-I':           df_updrs1,
    'UPDRS-II':          df_updrs2,
    'UPDRS-III':         df_updrs3,
    'UPDRS-IV':          df_updrs4,
    'MoCA':              df_moca,
    'PD Features':       df_pdfeat,
    'REM Sleep':         df_rem,
}

for name, df_clin in clinical_tables.items():
    before = len(df)
    df = df.merge(df_clin, on=JOIN_KEY, how='left')
    # display(f"{name}", df.columns.tolist())
    print(f"  Merged {name:<15} → {len(df)} rows (was {before})")

print(f"\nFinal shape: {df.shape}")
print(f"Rows retained: {len(df)}/{n_start} ({len(df)/n_start*100:.1f}%)")

  Merged UPDRS-I         → 2373 rows (was 2373)
  Merged UPDRS-II        → 2373 rows (was 2373)
  Merged UPDRS-III       → 2766 rows (was 2373)
  Merged UPDRS-IV        → 2766 rows (was 2766)
  Merged MoCA            → 2766 rows (was 2766)
  Merged PD Features     → 2766 rows (was 2766)
  Merged REM Sleep       → 2766 rows (was 2766)

Final shape: (2766, 312)
Rows retained: 2766/2373 (116.6%)
